In [1]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from timeseries import read_timeseries_csv
from scenarios.price_scenarios import build_price_matrix

zone = "centro_sud"

price_matrix = build_price_matrix()
price_matrix.head()

price_scenario Delayed transition                                      \
weather_year                 1991        1992        1993        1994   
contract_year                                                           
2026                   104.521571  104.982755  103.990135  104.093857   
2027                   102.100541  102.561726  101.569106  101.672827   
2028                    99.679512  100.140696   99.148076   99.251798   
2029                    97.258483   97.719667   96.727047   96.830769   
2030                    94.837453   95.298638   94.306018   94.409740   

price_scenario                                                              \
weather_year          1995        1996        1997        1998        1999   
contract_year                                                                
2026            103.284847  104.649355  103.784503  103.193555  103.306062   
2027            100.863818  102.228326  101.363474  100.772526  100.885032   
2028             98.442789   99.807296   98.942445   98.351496   98.464003   
2029             96.021759   97.386267   96.521415   95.930467   96.042974   
2030             93.600730   94.965238   94.100386   93.509438   93.621944   

price_scenario              ... Net Zero 2050                          \
weather_year          2000  ...          2014        2015        2016   
contract_year               ...                                         
2026            103.668238  ...    111.006854  110.544344  109.892894   
2027            101.247209  ...    111.738950  111.276440  110.624990   
2028             98.826179  ...    112.471046  112.008536  111.357086   
2029             96.405150  ...    113.203143  112.740633  112.089183   
2030             93.984121  ...    113.935239  113.472729  112.821279   

price_scenario                                                              \
weather_year          2017        2018        2019        2020        2021   
contract_year                                                                
2026            108.738251  110.885226  109.198881  109.718072  109.699545   
2027            109.470348  111.617323  109.930977  110.450168  110.431642   
2028            110.202444  112.349419  110.663073  111.182264  111.163738   
2029            110.934540  113.081515  111.395170  111.914360  111.895834   
2030            111.666636  113.813612  112.127266  112.646457  112.627931   

price_scenario                          
weather_year          2022        2023  
contract_year                           
2026            109.787155  109.378386  
2027            110.519251  110.110482  
2028            111.251348  110.842578  
2029            111.983444  111.574675  
2030            112.715540  112.306771  

[5 rows x 99 columns]

In [2]:
solar_dir = PROJECT_ROOT / "data" / "processed" / "solar" / zone

annual_solar_cf = {}
for year in range(1991, 2024):
    cf = read_timeseries_csv(solar_dir / f"solar_cf_{year}.csv")
    annual_solar_cf[year] = cf["solar_cf"].mean()

annual_solar_cf = pd.Series(annual_solar_cf)
annual_solar_cf.describe()

count    33.000000
mean      0.200528
std       0.004835
min       0.189947
25%       0.196989
50%       0.201503
75%       0.203541
max       0.210361
dtype: float64

In [3]:
STRIKE_PRICE_SOLAR = 56.83  # FER-X Transitorio weighted-average clearing price, EUR/MWh
HOURS_PER_YEAR = 8760

pap_payment_per_mw = STRIKE_PRICE_SOLAR * annual_solar_cf * HOURS_PER_YEAR
pap_payment_per_mw.describe()

count        33.000000
mean      99829.242612
std        2406.794154
min       94561.543522
25%       98067.324857
50%      100314.622290
75%      101328.737089
max      104724.086573
dtype: float64

In [4]:
from scenarios.load_profile import load_profile_for_archetype

annual_kwh = 20_000_000  # 20 GWh/year, a mid-size industrial buyer
load = load_profile_for_archetype("chemicals", annual_kwh)
load.sum() / 1000  # sanity check: should be close to 20,000 MWh

np.float64(20000.0)

In [5]:
annual_load_mwh = load.sum() / 1000
CONTRACTED_MW = 5

residual_mwh = (annual_load_mwh - annual_solar_cf * HOURS_PER_YEAR * CONTRACTED_MW).clip(lower=0)
residual_mwh.describe()

count       33.000000
mean     11216.853545
std        211.753841
min      10786.196853
25%      11084.925472
50%      11174.149015
75%      11371.870064
max      11680.314665
dtype: float64